In [1]:
from pathlib import Path
import os, sys, platform, subprocess



import torch, torchcodec

requested_profile = os.getenv("STT_PROFILE", "auto").strip().lower()
if requested_profile == "cuda" and not torch.cuda.is_available():
    active_profile = "cpu"
    print("STT_PROFILE=cuda requested, but CUDA is unavailable. Switched to CPU profile.")
elif requested_profile == "cpu":
    active_profile = "cpu"
elif requested_profile == "mps" and torch.backends.mps.is_available():
    active_profile = "mps"
elif requested_profile == "mps":
    active_profile = "cpu"
    print("STT_PROFILE=mps requested, but MPS is unavailable. Switched to CPU profile.")
else:
    active_profile = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

os.environ["STT_PROFILE"] = active_profile
print("exe:", sys.executable)
print("torch", torch.__version__, "torchcodec", torchcodec.__version__, "py", platform.python_version())
print("profile:", active_profile)
subprocess.run(["ffmpeg", "-version"], check=True)

exe: /workspaces/Speech_To_Text_Model/.venv/bin/python
torch 2.11.0+cpu torchcodec 0.11.0+cpu py 3.12.1
profile: cpu
ffmpeg version 6.1.1-3ubuntu5 Copyright (c) 2000-2023 the FFmpeg developers
built with gcc 13 (Ubuntu 13.2.0-23ubuntu3)
configuration: --prefix=/usr --extra-version=3ubuntu5 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --disable-omx --enable-gnutls --enable-libaom --enable-libass --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libglslang --enable-libgme --enable-libgsm --enable-libharfbuzz --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-lib

CompletedProcess(args=['ffmpeg', '-version'], returncode=0)

In [6]:
import datasets

dataset = datasets.load_dataset(
    "m-aliabbas/idrak_timit_subsample1",
    split="train"
)
print(dataset.shape)


(1296, 2)


In [5]:
dataset_test = datasets.load_dataset(
    "m-aliabbas/idrak_timit_subsample1",
    split="test"
)
print(dataset_test.shape)

(324, 2)


In [ ]:
from IPython.display import display
df = dataset.to_pandas()
print(f"Dataset rows: {len(df)}, columns: {len(df.columns)}")
display(df.head(20))

In [ ]:
# Install dependencies as needed:
# pip install kagglehub[pandas-datasets]
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Set the path to the file you'd like to load
file_path = ""

# Load the latest version
df = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "mozillaorg/common-voice",
  file_path,
  # Provide any additional arguments like 
  # sql_query or pandas_kwargs. See the 
  # documenation for more information:
  # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
)

print("First 5 records:", df.head())

In [ ]:
print(dataset.column_names)

In [ ]:
data = next(iter(dataset))
audio = data["audio"]["array"]
sample_rate = data["audio"]["sampling_rate"]
transcript = data["transcription"]

In [ ]:
data

In [ ]:
print(audio.shape)
print(sample_rate)
print("Audio length(s): ", audio.size / sample_rate)
print(transcript)

In [ ]:
from IPython.display import Audio
Audio(audio, rate=sample_rate)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

start_time_ms = 0
end_time_ms = 1000

start_idx = int(start_time_ms * sample_rate/ 1000)
end_idx = int(end_time_ms * sample_rate/ 1000)

audio_np = audio[start_idx:end_idx]

time_ms = np.arange(len(audio_np)) * (1000 / sample_rate)


plt.figure(figsize=(15, 5))
plt.plot(time_ms, audio_np)
plt.xlabel('Time(ms)')
plt.ylabel('Amplitude')
plt.title('Audio Waveform')
plt.grid(True)
plt.show()

print(f"Audio length: {len(audio_np) / sample_rate * 1000:.2f} ms")


In [ ]:
from tokenizers import Tokenizer, models, pre_tokenizers, decoders

def get_tokenizer(save_path="tokenizer.json"):
    tokenizer = Tokenizer(models.BPE())
    tokenizer.add_special_tokens(["☐"])
    tokenizer.add_tokens(list("ABCDEFGHIJKLMNOPQRSTUVWXYZ '"))

    tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel()
    tokenizer.decoder = decoders.ByteLevel()
    tokenizer.blank_token = tokenizer.token_to_id("☐")
    tokenizer.save(save_path)
    return tokenizer

In [ ]:
tokenizer = get_tokenizer()

In [ ]:
sorted(tokenizer.get_vocab().items(), key=lambda x:x[1])

In [ ]:
transcript = transcript.upper()
input_ids = tokenizer.encode(transcript)

In [ ]:
transcript

In [ ]:
input_ids.ids